# Training on BirdNET embeddings

## Split dataset and giving opensoundscape format

In [5]:
import pandas as pd

# Load the clip labels created from Process_boxes_to_classifier_labels.ipynb
clip_labels = pd.read_csv('/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/clip_labels_3s.csv')
# , index_col=[0,1,2]

print(f"Loaded {len(clip_labels)} clips with {len(clip_labels.columns)} call types")
print(f"Call types: {list(clip_labels.columns)}")
clip_labels.head()

Loaded 2201 clips with 14 call types
Call types: ['file', 'start_time', 'end_time', 'Cheer', 'A', 'Check', 'B', 'K', 'C', 'Chits', 'D', 'J', 'G', 'E']


,file,start_time,end_time,Cheer,A,Check,B,K,C,Chits,D,J,G,E
0,/mnt/class_data/Shelby/One_Minute_Audio/SL17a_...,0.0,3.0,False,0,False,False,False,0,False,0,False,False,0
1,/mnt/class_data/Shelby/One_Minute_Audio/SL17a_...,3.0,6.0,False,0,False,False,False,0,False,0,False,False,0
2,/mnt/class_data/Shelby/One_Minute_Audio/SL17a_...,6.0,9.0,False,0,False,False,False,0,False,0,False,False,0
3,/mnt/class_data/Shelby/One_Minute_Audio/SL17a_...,9.0,12.0,False,0,False,False,False,0,False,0,False,False,0
4,/mnt/class_data/Shelby/One_Minute_Audio/SL17a_...,12.0,15.0,False,0,False,False,False,0,False,0,False,False,0


In [ ]:
import os
import pandas as pd

# Path to YOLO dataset split files (now with updated trial allocation)
split_dir = '/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/YOLO/dataset_split/'

# Read the split files and extract unique trial identifiers
def extract_trial_ids(txt_file):
    """Extract unique trial identifiers from YOLO split file"""
    with open(txt_file, 'r') as f:
        paths = [line.strip() for line in f]
    
    # Extract trial IDs from image filenames
    # Format: .../AZ02_Trial_1_trim_spec_00-05.png -> AZ02_Trial_1_trim
    trial_ids = set()
    for path in paths:
        filename = os.path.basename(path)
        # Remove _spec_XX-XX.png to get trial ID
        trial_id = filename.rsplit('_spec_', 1)[0]
        trial_ids.add(trial_id)
    
    return trial_ids

# Extract trial IDs for each split
test_trials = extract_trial_ids(os.path.join(split_dir, 'test.txt'))
val_trials = extract_trial_ids(os.path.join(split_dir, 'validate.txt'))
train_trials = extract_trial_ids(os.path.join(split_dir, 'train.txt'))

print(f"Test trials: {sorted(test_trials)}")
print(f"Validation trials: {sorted(val_trials)}")
print(f"Train trials ({len(train_trials)} total): {sorted(list(train_trials)[:5])}...")

# Create masks based on trial IDs in the audio file paths
def create_mask(clip_labels, trial_ids):
    """Create boolean mask for clips from specified trial IDs"""
    return clip_labels.reset_index()["file"].apply(
        lambda x: any(trial_id in x for trial_id in trial_ids)
    ).values

# Split the data
mask_test = create_mask(clip_labels, test_trials)
test_set = clip_labels[mask_test]

mask_val = create_mask(clip_labels, val_trials)
val_set = clip_labels[mask_val]

mask_train = create_mask(clip_labels, train_trials)
train_set = clip_labels[mask_train]

# Verify the split
print(f"\nSplit summary:")
print(f"Training set: {len(train_set)} clips")
print(f"Validation set: {len(val_set)} clips")
print(f"Test set: {len(test_set)} clips")
print(f"Total: {len(train_set) + len(val_set) + len(test_set)} clips")
print(f"Original total: {len(clip_labels)} clips")

,start_time,end_time,calltype_A,calltype_B,calltype_C,calltype_Check,calltype_Cheer,calltype_Chits,calltype_D,calltype_E,calltype_G,calltype_J,calltype_K
file,,,,,,,,,,,,,
/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/SL07_Trial_1_trim_37.44715902_37.57169857_D.WAV,0.0,3.0,False,False,False,False,False,False,True,False,False,False,False
/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/SL10_Trial_1_trim_47.70134954_48.36720022_Cheer.WAV,0.0,3.0,False,False,False,False,True,False,False,False,False,False,False
/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/AZ13_Trial_3_trim_21.29153733_21.35859514_Check.WAV,0.0,3.0,False,False,False,True,False,False,False,False,False,False,False
/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/SL51_Trial_1_trim_57.29257191_58.21124032_Cheer.WAV,0.0,3.0,False,False,False,False,True,False,False,False,False,False,False
/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/AZ15_Trial_1_trim_17.73289918_17.82390596_Check.WAV,0.0,3.0,False,False,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/EF04_Trial_3_trim_18.05885722_18.41862923_G.WAV,0.0,3.0,False,False,False,False,False,False,False,False,True,False,False
/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/AZ13_Trial_1_trim_41.66087247_41.73991969_Check.WAV,0.0,3.0,False,False,False,True,False,False,False,False,False,False,False
/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/Call_audios/padded_3s/SL07_Trial_1_trim_1.432204814_1.504054554_Check.WAV,0.0,3.0,False,False,False,True,False,False,False,False,False,False,False


In [ ]:
# Save .csv tables of the training, validation, and test sets
os.makedirs("./annotated_data", exist_ok=True)
train_set.to_csv("./annotated_data/train_set.csv")
val_set.to_csv("./annotated_data/val_set.csv")
test_set.to_csv("./annotated_data/test_set.csv")

print(f"\nSaved splits to ./annotated_data/")

optional: upsampling or downsampling

In [3]:
train_set.head()

,file,start_time,end_time,Cheer,A,Check,B,K,C,Chits,...,D,J,O,I,F,G,H,E,N,L
20,/mnt/class_data/Shelby/One_Minute_Audio/AZ02_T...,0.0,3.0,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
21,/mnt/class_data/Shelby/One_Minute_Audio/AZ02_T...,3.0,6.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
22,/mnt/class_data/Shelby/One_Minute_Audio/AZ02_T...,6.0,9.0,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
23,/mnt/class_data/Shelby/One_Minute_Audio/AZ02_T...,9.0,12.0,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
24,/mnt/class_data/Shelby/One_Minute_Audio/AZ02_T...,12.0,15.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [4]:
#can we check how many of each call type are in the train, val, and test sets?
print("Training set call type counts:")
print(train_set.sum())

print("Validation set call type counts:")
print(val_set.sum())

print("Test set call type counts:")
print(test_set.sum())

Training set call type counts:
file          /mnt/class_data/Shelby/One_Minute_Audio/SL17a_...
start_time                                              44460.0
end_time                                                49140.0
Cheer                                                       376
A                                                            31
Check                                                       765
B                                                            16
K                                                            62
C                                                            61
Chits                                                       116
D                                                            98
J                                                            14
G                                                            37
E                                                           124
dtype: object
Validation set call type counts:
file          /mnt/class_d

### resample

In [46]:
# resample train set by upsampling minority classes
n_samples_per_class = 4
seed = 42

resampled_train_set = pd.DataFrame(columns=train_set.columns)
for class_label in train_set.columns:
    # print(f"Processing class: {class_label}")
    if class_label != "start_time" and class_label != "end_time" and class_label != "file": 
        class_clips = train_set[train_set[class_label] == 1]
        if len(class_clips) == 0:
            continue
        
        n_repeats = n_samples_per_class // len(class_clips)
        n_extra = n_samples_per_class % len(class_clips)
        
        resampled_class_clips = pd.concat([class_clips] * n_repeats + [class_clips.sample(n_extra, replace=True, random_state=seed)])
        resampled_train_set = pd.concat([resampled_train_set, resampled_class_clips])

resampled_train_set = resampled_train_set.drop_duplicates().reset_index(drop=True)
resampled_train_set.set_index(['file', 'start_time', 'end_time'], inplace=True)

In [48]:
resampled_train_set.sum(axis=0)

Cheer    13
A         3
Check    36
B         5
K        10
C         4
Chits    10
M         8
Growl     4
D         9
J         5
O         4
I         4
F         7
G         4
H         4
E         6
N         4
L         4
dtype: object

In [45]:
resampled_train_set.to_csv("resampled_train_set.csv")
val_set.to_csv("validation_set.csv")